# Harmonia — input variability flips the safety classification

**The headline analysis, reproduced from the dataset + the reference kernel.**

> Harmonia is **NOT** a clinical tool and **NOT** a regulatory safety
> determination. It reports a torsade-risk-metric *distribution* and a
> classification-flip frequency — never a bare "safe/unsafe" verdict
> (spec.md §10).

This notebook is executed in CI under [`nbmake`](https://github.com/treebeardtech/nbmake);
if it runs clean, the numbers below are a faithful, deterministic (seeded)
projection of the curated data. It mirrors `docs/make_figures.py`, which writes
the README figures.

In [ ]:
import numpy as np
import harmonia
from harmonia.performance import score

ds = harmonia.load()
print(harmonia.CLINICAL_USE)
print(f"dataset {harmonia.__version__}: {len(ds.records)} records, "
      f"{len(ds.drug_references)} CiPA compounds")

## 1. One record: variability is first-class

A channel-block record carries the *multiple* lab measurements of the same IC50
and the computed inter-source spread, not a single number.

In [ ]:
b = ds["channel_block.dofetilide.ikr"]
print(f"{b.id}  tier={b.tier}")
print(f"  max block observed : {b.assay_context.max_block_observed_percent}%  "
      f"(>60 => IC50 identifiable)")
print(f"  fold-range         : {b.variability.fold_range}  "
      f"across {b.variability.n_sources} sources")
for sv in b.source_values:
    print(f"    {sv['ic50_nm']:>6} nM  {sv['platform']:<10} {sv['citation']}")

## 2. The headline: input variability flips the classification

Pick a drug; Harmonia pulls the spread of published IC50s per channel, propagates
it by Monte-Carlo through the chosen AP model, and reports the **qNet
distribution** and how often the high/intermediate/low call **flips**. A
near-pure hERG blocker (dofetilide) lands tightly in HIGH; a balanced
multichannel blocker (verapamil) straddles the low/intermediate boundary.

In [ ]:
for drug in ["dofetilide", "verapamil"]:
    a = harmonia.assess(ds, drug, ap_model="cipaordv1.0", n_mc=120, seed=0)
    q = a.qnet_distribution
    lo, hi = a.flip_ci   # Wilson 95% CI of the flip frequency (v0.7)
    mix = {k: round(v, 2) for k, v in a.classification_distribution.items() if v}
    print(f"{drug:<12} tier={a.tier}  point={a.classification.upper():<13}"
          f"qNet={a.qnet:.3f}  flip={a.classification_flip_frequency:.0%} "
          f"(95% CI {lo:.0%}-{hi:.0%}, 120 draws)")
    print(f"             qNet MC: median={np.median(q):.3f} "
          f"[p5={np.percentile(q, 5):.3f}, p95={np.percentile(q, 95):.3f}]  "
          f"class mix={mix}")

In [ ]:
# Sanity assertions (these make the notebook a CI test, not just a demo):
dof = harmonia.assess(ds, "dofetilide", ap_model="cipaordv1.0", n_mc=120, seed=0)
ver = harmonia.assess(ds, "verapamil", ap_model="cipaordv1.0", n_mc=120, seed=0)
assert dof.classification == "high"
assert dof.classification_flip_frequency < ver.classification_flip_frequency, (
    "a balanced multichannel blocker must be less classification-stable "
    "than a near-pure hERG blocker")
# v0.7: every reported flip frequency carries a Wilson 95% CI that brackets it
# and is wider for the less-resolved (more variable) drug.
for a in (dof, ver):
    lo, hi = a.flip_ci
    assert lo <= a.classification_flip_frequency <= hi <= 1.0
print("OK: dofetilide stable in HIGH; verapamil flips more often; "
      "each flip frequency carries a Wilson 95% CI.")

## 3. The safety call also moves with the AP-model variant

`flip_view` runs the same drug through the ORd, CiPAORd v1.0, and ToR-ORd
variants. For a boundary drug like verapamil the *structural* choice of model can
move the point classification — another axis of the same honesty.

In [ ]:
cmp = harmonia.flip_view(ds, "verapamil", n_mc=80, seed=0)
print(cmp.summary())
print("\npoint class by model:", cmp.flip_by_model)
print("stable across model variants:", cmp.stable_across_models)

## 4. Recorded classification performance (honest numbers)

`harmonia.performance.score` scores the reduced kernel's qNet classification
against the CiPA expert labels and prints the full confusion matrix. These are a
*methodology demonstrator*, not a qualified classifier — the durable
contribution is the flip-frequency machinery above, correct regardless of
absolute accuracy.

In [ ]:
rep = score(ds, ap_model="cipaordv1.0", cipa_set="all")
print(rep)
# Never a two-category error (calling a high-risk drug low, or vice versa):
assert rep.adjacent_accuracy() == 1.0, "a two-category miss would be a safety-screen failure"
print("\nOK: within-one-category accuracy is 100% (no catastrophic miss).")

## 5. Unidentifiable inputs are stated, not imputed

Where the maximum block observed is below ~60%, the IC50 is unidentifiable; the
channel is **excluded** from the simulation and caps any assessment that touches
it at Tier D — never silently point-estimated.

In [ ]:
rano = harmonia.assess(ds, "ranolazine", ap_model="cipaordv1.0", n_mc=0)
print("excluded (unidentifiable) channels:", rano.excluded_channels)
print("propagated tier:", rano.tier)
for w in rano.warnings:
    print("  warning:", w)